In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
from sklearn.metrics import classification_report
import pandas as pd
import requests
import re
import numpy as np
import warnings
warnings.filterwarnings('ignore')


def get_answers(text):
    if re.fullmatch("d+", text):
        matches = re.findall("\d+", text)
    elif re.fullmatch("[\d+, ]+", text):
        matches = re.findall("\d+", text)
    else:
        matches = re.findall("\d+", text)[-1:]
    return [int(m) for m in matches]


def default_evaluation(true_labels, pred_labels):
    result = dict()
    result['precision'] = precision_score(true_labels, pred_labels)
    result['recall'] = recall_score(true_labels, pred_labels)
    result['f1'] = f1_score(true_labels, pred_labels)
    result['accuracy'] = accuracy_score(true_labels, pred_labels)
    return result


def get_sampled_results(preds, df, report=False):
    new_df = {"sample_id": [], "prediction": []}

    for _, row in preds.iterrows():
        for _, id in row['match_dict'].items():
            new_df["sample_id"].append(id)
            new_df["prediction"].append(id in row['prediction_sample_id'])

    result = pd.DataFrame(data=new_df)
    result = result.sort_values(by=["sample_id"])
    result.prediction = result.prediction.astype(np.int32)
    df = df.sort_values(by=["sample_id"])

    if report:
        true_labels = df['correct'].astype(np.int32).values
        pred_labels = result['prediction'].astype(np.int32).values
        print(classification_report(pred_labels, true_labels))
        metrics = default_evaluation(true_labels, pred_labels)
        print(metrics)
        return result, metrics
    
    return result


key = ''
def wikimedia_pagereviews(label, start_data="20230101", end_date="20230901"):
    label = label[0].upper() + label[1:]
    if '/' in label:
        test_lbl = label.replace('/', '%2F')
    elif '?' in label:
        test_lbl = label.replace('?', '%3F')
    else:
        test_lbl = '_'.join(label.split(' '))
        
    url = "https://wikimedia.org/api/rest_v1/metrics/pageviews/per-article/en.wikipedia.org/all-access/all-agents/"
    url += f"{test_lbl}/monthly/{start_data}/{end_date}"
    headers = {
        'accept': 'application/json',
        'Authorization': key,
        'User-Agent': '2368ba8df0e5f6b69af8027312c507c654c234c8'
    }

    response = requests.get(
        url = url,
        headers=headers,
    )   
    
    return response.json()['items'][0]['views']